In [2]:
from pathlib import Path
from eeg_music.data import EEGMusicDataset, MappedDataset, MelParams, TrialData, WavRAW, prepare_trial, wavraw_to_melspectrogram


bcmi_ds = EEGMusicDataset.load_ondisk(
    Path("./datasets/bcmi_preprocessed/bcmi_pre_60ch/")
  )
musing_ds = EEGMusicDataset.load_ondisk(
    Path("./datasets/musing_preprocessed/musing_pre_60ch/")
  )

/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [ ]:

from eeg_music.data import MusingMusicId, MusingMusicIdData


def map_bcmi_to_id(sample, ds):
  id1 = ds.df.index.get_loc((sample.dataset, sample.subject, sample.session, sample.run, sample.trial_id))
  return TrialData(
    dataset=sample.dataset,
    subject=sample.subject,
    session=sample.session,
    run=sample.run,
    trial_id=sample.trial_id,
    music_filename=sample.music_filename,
    music_data=MusingMusicIdData(MusingMusicId(id1)),
    eeg_data=sample.eeg_data,
  )

mapped_bcmi_baseline = MappedDataset(bcmi_ds, lambda t:  map_bcmi_to_id(t, bcmi_ds))

mapped_bcmi_baseline[0]

TrialData(dataset='bcmi-training', subject='08', session='3', run='1', trial_id='trial_1', music_filename=MusicFilename(filename='1-6_3_first.wav'), eeg_data=OnDiskArrayEeg(filepath=PosixPath('datasets/bcmi_preprocessed/bcmi_pre_60ch/eeg/bcmi-training/08/3/1/trial_1/eeg.npz')), music_data=MusingMusicIdData(music_id=MusingMusicId(song_id=0)))

In [12]:
mapped_bcmi_baseline.save(Path("datasets/bcmi_preprocessed/bcmi_ids_60ch"))

In [5]:
sample = bcmi_ds[17]
bcmi_ds.df.index.get_loc((sample.dataset, sample.subject, sample.session, sample.run, sample.trial_id))

17

In [ ]:

from eeg_music.data import MusingMusicId, MusingMusicIdData
from eeg_music.emotion_utils import parse_music_emotion

def map_bcmi_to_id(sample, ds):
  emotion = parse_music_emotion(sample.music_filename.filename, sample.dataset) - 1 # type: ignore
  return TrialData(
    dataset=sample.dataset,
    subject=sample.subject,
    session=sample.session,
    run=sample.run,
    trial_id=sample.trial_id,
    music_filename=sample.music_filename,
    music_data=MusingMusicIdData(MusingMusicId(emotion)),
    eeg_data=sample.eeg_data,
  )

mapped_bcmi_emotion = MappedDataset(bcmi_ds, lambda t:  map_bcmi_to_id(t, bcmi_ds))

mapped_bcmi_emotion[1]

TrialData(dataset='bcmi-training', subject='08', session='3', run='1', trial_id='trial_2', music_filename=MusicFilename(filename='1-6_3_second.wav'), eeg_data=OnDiskArrayEeg(filepath=PosixPath('datasets/bcmi_preprocessed/bcmi_pre_60ch/eeg/bcmi-training/08/3/1/trial_2/eeg.npz')), music_data=MusingMusicIdData(music_id=MusingMusicId(song_id=5)))

In [15]:
mapped_bcmi_emotion.save(Path("./datasets/bcmi_preprocessed/bcmi_emotion_60ch/"))

In [17]:
from eeg_music.onset_conversion import trial_wavraw_to_noteonsets

note_bcmi_ds = MappedDataset(bcmi_ds, trial_wavraw_to_noteonsets)
note_bcmi_ds[0]

TrialData(dataset='bcmi-training', subject='08', session='3', run='1', trial_id='trial_1', music_filename=MusicFilename(filename='1-6_3_first.wav'), eeg_data=OnDiskArrayEeg(filepath=PosixPath('datasets/bcmi_preprocessed/bcmi_pre_60ch/eeg/bcmi-training/08/3/1/trial_1/eeg.npz')), music_data=NoteOnsets(onset_times=array([1.1609977e-02, 9.2879817e-02, 9.2879820e-01, 1.8111565e+00,
       2.5890250e+00, 3.4713833e+00, 4.4582314e+00, 5.4450793e+00,
       6.4435372e+00, 7.1633558e+00, 8.2198639e+00, 8.9977322e+00,
       9.9729710e+00, 1.0843719e+01, 1.1726077e+01, 1.2492335e+01,
       1.3363084e+01, 1.4361542e+01, 1.5348390e+01, 1.6335238e+01,
       1.7066668e+01, 1.8111565e+01, 1.8912653e+01, 1.9876282e+01,
       2.0003990e+01], dtype=float32), sample_rate=44100, duration_seconds=20.0))

In [20]:
note_bcmi_ds.save(Path("./datasets/bcmi_preprocessed/bcmi_notes_60ch"))

OSError: [Errno 28] No space left on device: 'datasets/bcmi_preprocessed/bcmi_notes_60ch/eeg/bcmi-training/10/2/2/trial_23'

In [ ]:
musing_ds[0]

MusingMusicIdData(music_id=MusingMusicId(song_id=1))